## Mega-Sena – Geração de Apostas com Redes Neurais

Este projeto utiliza Redes Neurais Densas (Keras/TensorFlow) para analisar
padrões históricos dos sorteios da Mega-Sena e gerar sugestões de apostas
baseadas em probabilidades aprendidas a partir dos dados.

### Abordagem
- Representação dos sorteios em formato multi-hot (60 dimensões)
- Treinamento de uma rede neural com arquitetura densa
- Uso de ativação sigmoid para modelar probabilidades independentes
- Geração de apostas com base no ranking probabilístico dos números

### ⚠️ Observação Importante
A Mega-Sena é um processo aleatório. Este projeto **não tem o objetivo de
prever sorteios**, mas sim demonstrar técnicas de:
- modelagem probabilística
- redes neurais
- análise exploratória
- construção de pipelines em Data Science


In [ ]:
#Mega-Sena – Geração de Apostas com Redes Neurais
import pandas as pd
import ssl
import random

ano = 2025

def xls_resultados(url):
    """
    Obtém os dados de todos os sorteios da Mega Sena.
    
    :param url: Endereço para obter o arquivo XLS
    com os dados dos sorteios da Mega Sena
    
    :returns: DataFrame com os dados do arquivo XLS
    """
    # Configura o contexto SSL para ignorar verificação de certificado
    ssl._create_default_https_context = ssl._create_unverified_context
    
    # Lê os dados diretamente da URL
    dados = pd.read_excel(url)
    return dados

# URL do arquivo XLS
URL = "https://servicebus2.caixa.gov.br/portaldeloterias/api/resultados/download?modalidade=Mega-Sena"

# Obtém os dados do arquivo XLS
base = xls_resultados(URL)

# Remove dados duplicados com base na coluna 'Concurso'
base = base.drop_duplicates('Concurso')

# Renomeia as colunas do DataFrame
colunas = {'Bola1': 'B1', 'Bola2': 'B2', 'Bola3': 'B3', 'Bola4': 'B4', 'Bola5': 'B5', 'Bola6': 'B6'}
base.rename(columns=colunas, inplace=True)

# Filtrar os concursos do ano escolhido
base['Data do Sorteio'] = pd.to_datetime(base['Data do Sorteio'], dayfirst=True, errors='coerce')  # Converte a coluna de datas para o formato datetime
base_ano = base[base['Data do Sorteio'].dt.year == ano]  # Filtra apenas os concursos do ano que foi escolhido

# Contar a frequência de cada número nos concursos escolhido
frequencias = pd.Series(0, index=range(1, 61))  # Inicializa uma série com 0 para cada número de 1 a 60

# Conta a ocorrência de cada número do ano escolhido
for coluna in ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']:
    frequencias += base_ano[coluna].value_counts().reindex(range(1, 61), fill_value=0)

# Calcula a probabilidade de cada número com base nas frequências
total_sorteios = len(base_ano)
probabilidades = frequencias / (total_sorteios * 6)  # 6 números por sorteio

# Exibe os números mais prováveis
print(f"Frequência dos números nos concursos de : {ano}")
print(frequencias.sort_values(ascending=False))

print(f"\nProbabilidades dos números com base nos concursos de: {ano}")
print(probabilidades.sort_values(ascending=False))

# Cria uma coluna que representa a combinação sorteada como uma tupla ordenada
base['Combinacao'] = base[['B1', 'B2', 'B3', 'B4', 'B5', 'B6']].apply(lambda x: tuple(sorted(x)), axis=1)

# Função para gerar uma combinação levando em conta as probabilidades sem repetição
def gerar_combinacao_com_probabilidades(probabilidades):
    numeros = list(range(1, 61))
    combinacao = random.choices(numeros, weights=probabilidades, k=6)  # Gera 6 números com base nas probabilidades
    
    # Garantir que a combinação não tenha números repetidos
    combinacao_unica = list(set(combinacao))  # Remove duplicatas
    while len(combinacao_unica) < 6:  # Caso tenha menos que 6 números únicos
        combinacao_unica.append(random.choices(numeros, weights=probabilidades, k=1)[0])
    return tuple(sorted(combinacao_unica))  # Retorna como tupla ordenada


# Função para gerar uma combinação única (que ainda não saiu nos concursos anteriores)
def gerar_combinacao_unica(probabilidades, combinacoes_anteriores):
    nova_combinacao = gerar_combinacao_com_probabilidades(probabilidades)
    
    # Continua gerando combinações até encontrar uma que ainda não saiu
    while nova_combinacao in combinacoes_anteriores:
        nova_combinacao = gerar_combinacao_com_probabilidades(probabilidades)
    
    return nova_combinacao

# Obtém todas as combinações anteriores (incluindo concursos de todos os anos)
combinacoes_anteriores = set(base['Combinacao'])

# Função para gerar várias combinações únicas
def gerar_multiplas_combinacoes_unicas(n, probabilidades, combinacoes_anteriores):
    combinacoes_unicas = []
    
    while len(combinacoes_unicas) < n:
        nova_combinacao = gerar_combinacao_unica(probabilidades, combinacoes_anteriores)
        
        # Verifica se a nova combinação já foi gerada antes
        if nova_combinacao not in combinacoes_unicas:
            combinacoes_unicas.append(nova_combinacao)
            combinacoes_anteriores.add(nova_combinacao)  # Adiciona a nova combinação ao histórico
    
    return combinacoes_unicas

# Gerar qtde de  combinações únicas
novas_combinacoes = gerar_multiplas_combinacoes_unicas(40, probabilidades, combinacoes_anteriores)

# Exibe as novas combinações geradas
print("\nNovas combinações geradas (que ainda não saíram):")
for combinacao in novas_combinacoes:
    print(combinacao)

In [ ]:
print('len(base_ano)', len(base_ano))
print('min data', base_ano['Data do Sorteio'].min())
print('max data', base_ano['Data do Sorteio'].max())
print(base_ano['Data do Sorteio'].head(10))
print("Datas NaT:", base['Data do Sorteio'].isna().sum())

base_ano = base[base['Data do Sorteio'].dt.year == ano]



In [ ]:
#!pip install tensorflow
import tensorflow as tf
print(tf.__version__)


In [ ]:
#Exemplo de Aplicação com Keras/TensorFlow (Redes Neurais Profundas):
#Passo 1: Preparação dos Dados
#Usaremos a estrutura dos sorteios passados, transformando-a em um formato adequado para alimentar a rede neural.

import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Supondo que 'base_ano' seja o DataFrame com os dados dos sorteios do ano corrente 
# Prepare os dados de entrada (X) e saída (y) para a rede neural

# Números de 1 a 60
numeros = list(range(1, 61))

# Prepare os dados de entrada (X)
X = []

# Criando o vetor de entrada para cada sorteio (6 números por sorteio)
for _, row in base_ano.iterrows():
    sorteio = [0] * 60  # Inicializa com 60 zeros
    for i in range(1, 7):  # Para as 6 bolas sorteadas
        sorteio[row[f'B{i}'] - 1] = 1  # Marca o número como sorteado
    X.append(sorteio)

X = np.array(X)
# Convertendo explicitamente para float64
X = X.astype(np.float64)

# Agora, para 'y', criaremos uma variável alvo que tem 144 amostras, cada uma representando um sorteio
y = []

# Para cada sorteio, criamos um vetor de 6 elementos, indicando os números sorteados
for _, row in base_ano.iterrows():
    sorteio_y = [0] * 60  # Inicializa com 60 zeros
    for i in range(1, 7):  # Para as 6 bolas sorteadas
        sorteio_y[row[f'B{i}'] - 1] = 1  # Marca o número como sorteado
    y.append(sorteio_y)

y = np.array(y)  # Agora 'y' é uma matriz de 144 (amostras) x 60 (números)

# Verifique as dimensões de X e y
print(X.shape)  #(144, 60) no momento do teste
print(y.shape)  

# Divisão dos dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizando os dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [ ]:
#Passo 2: Definindo a Rede Neural
#Agora, vamos construir um modelo de rede neural profunda (DNN).

# Definir a arquitetura do modelo
modelo = Sequential()

# Camada de entrada
modelo.add(Input(shape=(60,)))

# Adicionando camadas densas (fully connected layers)
modelo.add(Dense(128, activation='relu'))  # Primeira camada oculta
modelo.add(Dense(64, activation='relu'))  # Segunda camada oculta
modelo.add(Dense(32, activation='relu'))  # Terceira camada oculta

modelo.add(Dense(60, activation='sigmoid'))  # Camada de saída com ativação sigmoid para classificação binária

# Compilando o modelo
modelo.compile(loss='binary_crossentropy', optimizer='adam', metrics=['binary_accuracy']) 
# Treinando o modelo
modelo.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))


In [ ]:
'''
binary_accuracy (mais adequada aqui ✅)
Avalia posição por posição
Cada uma das 60 saídas é tratada como:
uma classificação binária independente
Combina perfeitamente com:
sigmoid
binary_crossentropy
problemas multi-label

📌 Em outras palavras:
“De todos os 60 números, quantos o modelo acertou como 0 ou 1?”
'''

In [ ]:
#Passo 3: Avaliação e Previsão
#Após o treinamento, podemos avaliar a performance do modelo e realizar previsões.

# Avaliar o modelo
perda, acuracia = modelo.evaluate(X_test, y_test)
print(f"Acurácia do modelo: {acuracia:.2f}")

# Fazer previsões
y_pred = modelo.predict(X_test)
print(f"y_pred : {y_pred.shape}")


In [ ]:
# Previsões 
def gerar_jogos_probabilisticos(y_pred, n_jogos=10):
    numeros = np.arange(1, 61)
    jogos = []

    for i in range(min(n_jogos, len(y_pred))):
        probs = y_pred[i]
        probs = probs / probs.sum()  # normaliza

        jogo = np.random.choice(
            numeros,
            size=6,
            replace=False,
            p=probs
        )

        jogos.append(sorted(jogo))

    return jogos

jogos_gerados = gerar_jogos_probabilisticos(
    y_pred,
    n_jogos=10
)

print("\nJogos sugeridos pelo modelo:")
for i, jogo in enumerate(jogos_gerados, 1):
    print(f"Jogo {i}: {jogo}")
'''
Converter sorteio em vetores de 60 posições
Treinar uma rede que aprende padrões
Gerar probabilidade para cada número
Criar jogos completos (6 números) respeitando essas probabilidades
'''

In [ ]:
#Top 6
numeros = np.arange(1, 61)

y_pred = modelo.predict(X_test)

# Exibir as 10 primeiras previsões
for i in range(10):
    print(f"Sorteio {i+1}:")

    probs = y_pred[i]

    # pega os índices dos 6 maiores valores
    top6_idx = np.argsort(probs)[-6:][::-1]
    numeros_previstos = numeros[top6_idx]

    print(f"Previsão (6 números): {sorted(numeros_previstos)}")


In [ ]:
# Mantendo a ideia probabilistica
for i in range(10):
    print(f"Sorteio {i+1}:")

    probs = y_pred[i]
    probs = probs / probs.sum()

    numeros_previstos = np.random.choice(
        numeros,
        size=6,
        replace=False,
        p=probs
    )

    print(f"Previsão probabilística: {sorted(numeros_previstos)}")


In [ ]:
# Fazer previsões binárias
y_pred = modelo.predict(X_test)
y_pred_binario = (y_pred > 0.5).astype(int)  # Convertendo as probabilidades em previsões binárias

# Exibir as 10 primeiras previsões
for i in range(10):
    print(f"Sorteio {i+1}:")
    previsao = y_pred_binario[i]
    numeros_sorteados = [numeros[j] for j in range(60) if previsao[j] == 1]
    print(f"Previsão: {numeros_sorteados}")

'''
qualquer número com probabilidade > 0.5 vira 1
o restante vira 0
(threshold fixo 0.5)
'''
